# Scholl et al. (2015) on CRCNS pvc-10

Paper: Scholl, Pattadkal, Dilly, Priebe and Zemelman (2015), "Local Integration Accounts for Weak
Selectivity of Mouse Neocortical Parvalbumin Interneurons", Neuron 87:424-436,
doi:10.1016/j.neuron.2015.06.030. Claims file: `claims/scholl-2015.yaml`, written from the version
of record (main text, Experimental Procedures and Supplemental Information legends).

Data set: CRCNS pvc-10, "Disparity selectivity in mouse V1 measured using two-photon calcium
imaging", doi:10.6080/K0GT5K3V, read anonymously from the public S3 bucket `crcnsarchive`.

Version. CRCNS data sets are not versioned, so the version is recorded as `unversioned`. The data
set as published is the folder `pvc-10/live/`, the one its file list (`live/filelist.txt`) and its
`live/checksums.md5` describe; every file there is listed with an MD5 sum. The bucket also holds,
under the same prefix, a folder `pvc-10/original/to_remove/` with processed per-cell MATLAB structs
that neither list describes. This attempt uses them, for the reasons given in the data check: they
are public and under the data set's own identifier, each file read is identified by the ETag the
listing gives (printed below), and for the 9 cells checked on one plane the stored traces equal the
mask means of the published movie.
Every tested claim rests on them, and on the assumption that their cells are the paper's.

Outcome in brief. The published folder holds the raw movies, stimulus tables and plane shifts of 6
PV-Cre;Ai14 mice, and no cell masks, PV+/PV- labels or per-cell measures. The unpublished structs
hold per-cell responses, DSI, preferences and positions for PV+ and PV- cells of mice 8, 9 and 10,
and PV+ cells of mouse 7. Seven claims are blocked (histology, CaMKIIa mice, neuropil patches, and
the counts of all identified cells, none of which is deposited). Eighteen are tested on 3 of the 6
animals (4 for the PV+ side of Figure 3). The cells are taken as the authors' `binocmonocinds`
lists, an assumption the data check explains; most outcomes that rest on it change under another
cell set, and the notebook shows them all.

| Claim | Outcome |
|---|---|
| fig1e-pv-tdtomato-colocalization | blocked, data_missing: no histology |
| cell-counts-responsive | blocked, data_missing: the identified cells of the six mice are not deposited |
| fig3abc-pv-more-binocular-fraction | reproduced (assumed) |
| fig3c-contralateral-bias | reproduced |
| fig3-pv-lower-monocularity | partially reproduced (assumed) |
| mi-per-animal | reproduced on 3 animals (assumed) |
| fig3def-pv-weaker-disparity-selectivity | partially reproduced (assumed) |
| mi-dsi-relation | not reproduced (assumed) |
| pv-larger-dff-low-dsi | not reproduced |
| figs3-pv-distinct-from-neuropil | blocked, data_missing: no neuropil patches, no non-responsive PV+ cells |
| fig4ag-salt-and-pepper | reproduced |
| fig4d-pv-od-matches-local-50um | not reproduced (assumed) |
| fig4f-pv-od-spatial-decay | partially reproduced |
| fig4f-od-length-constant | not reproduced |
| fig4ef-pvminus-od-unrelated | partially reproduced (assumed) |
| figs6ab-pvminus-pv-like-od | reproduced (assumed) |
| figs4-same-plane-control | not reproduced |
| figs5-neuropil-no-relation | blocked, data_missing: no neuropil patches |
| fig5-camkii-responsive | blocked, data_missing: no CaMKIIa animals |
| fig5e-camkii-od-unrelated | blocked, data_missing: no CaMKIIa animals |
| fig4i-pv-disparity-matches-local | not reproduced |
| fig4k-pv-disparity-length-constant | not reproduced |
| fig4jk-pvminus-disparity-weak | not reproduced |
| figs6cd-pvminus-pv-like-disparity | reproduced (assumed) |
| fig5f-camkii-disparity-unrelated | blocked, data_missing: no CaMKIIa animals |

The table gives the outcomes after the third verification round, which agreed with two changes
proposed after the second: fig4d-pv-od-matches-local-50um to not reproduced (its criterion has no
partial case, and the count fails under every reading), and fig4f-pv-od-spatial-decay to
partially reproduced (under readings 7b and 1b only the 50 um slope is significant).

In [ ]:
import collections
import hashlib
import resource
import time

import matplotlib.pyplot as plt
import numpy as np

import pvc10

T0 = time.time()
print("dataset", pvc10.DATASET, pvc10.VERSION, "doi", pvc10.DOI, "folder", pvc10.LIVE)

## Data check

### Files in the bucket

The data descriptor (live/docs, Version 0.8, April 8, 2016) says the data are from 6 animals in
folders mouse5 to mouse10, with green and red channel movies and a stimulus file per plane and a
stats file per mouse. It also says "The dataset provided here contains information about the
fluorescence traces, peak responses per condition, the measured binocular characteristics for each
cell like its disparity selectivity index, preferred relative disparity, the position of the cell in
the imaging volume". The listing below checks what is there.

In [ ]:
listing = pvc10.list_bucket()
by_folder = collections.defaultdict(lambda: [0, 0])
for k, s in listing:
    if k.endswith("/"):
        continue
    parts = k.split("/")
    folder = "/".join(parts[1:3])
    if parts[1] == "live" and parts[2] == "data":
        folder = "live/data/" + parts[3]
    by_folder[folder][0] += 1
    by_folder[folder][1] += s
print(f"{'folder':28s} {'files':>6s} {'GB':>8s}")
for f, (n, s) in sorted(by_folder.items()):
    print(f"{f:28s} {n:6d} {s / 1e9:8.2f}")
live_data = [(k, s) for k, s in listing if k.startswith(pvc10.LIVE + "data/") and not k.endswith("/")]
print("live/data total", round(sum(s for _, s in live_data) / 1e9, 1), "GB in", len(live_data), "files")

kinds = collections.Counter()
for k, _ in live_data:
    name = k.rsplit("/", 1)[1]
    kind = ("green" if name.endswith("green.mat") else "red" if name.endswith("red.mat")
            else "stim" if name.endswith("stim.mat") else "stats" if name.endswith("stats.mat") else "other")
    kinds[kind] += 1
print("live/data files by kind", dict(kinds))
print("files in live/ outside data/:", [k.split("live/")[1] for k, _ in listing
                                         if k.startswith(pvc10.LIVE) and not k.startswith(pvc10.LIVE + "data") and not k.endswith("/")])

In [ ]:
P = pvc10.planes()
md5 = pvc10.checksums()
print("planes per mouse:", {m: len(d) for m, d in P.items()}, "total", sum(len(d) for d in P.values()))
print("paper: 6 animals, 6 to 15 focal planes each, 150 to 460 um below the pia")
for m, ds in P.items():
    st = pvc10.stats_table(m)
    same = sorted(st[:, 0].astype(int).tolist()) == ds
    shifts = sorted({(float(a), float(b)) for a, b in st[:, 2:]})
    print(f"mouse{m}: depths {ds[0]} to {ds[-1]} um; stats depths match file names: {same}; "
          f"stimulus duration {sorted(set(st[:, 1]))} s; plane shifts (x, y um) {shifts}")
listed = set(md5)
missing_md5 = [k for k, _ in live_data if k.split("live/")[1] not in listed]
print("live/data files without an MD5 in checksums.md5:", len(missing_md5))

### Stimulus tables

Each stim file has two columns: the frame at which a stimulus starts and its condition (1 blank,
2 to 9 the binocular phases 0 to 315 deg, 10 ipsilateral, 11 contralateral). The paper says 6 to
7 repeats per focal plane, a 1.5 to 2 s stimulus followed by a 3 s blank, and imaging at 30 Hz.

In [ ]:
stims = {}
rows = []
for m, ds in P.items():
    dur = pvc10.stats_table(m)[0, 1]
    for d in ds:
        s = pvc10.stim_table(m, d)
        stims[(m, d)] = s
        counts = np.bincount(s[:, 1], minlength=12)[1:]
        isi = np.diff(s[:, 0])
        rows.append((m, d, len(s), counts.min(), counts.max(), set(s[:, 1]) == set(range(1, 12)),
                     np.median(isi), isi.min(), isi.max(), dur))
print(f"{'mouse':>5s} {'depth':>5s} {'trials':>6s} {'rep min':>7s} {'rep max':>7s} {'11 conds':>8s} "
      f"{'ISI med':>7s} {'min':>4s} {'max':>5s} {'stim s':>6s}")
for r in rows:
    print(f"{r[0]:5d} {r[1]:5d} {r[2]:6d} {r[3]:7d} {r[4]:7d} {str(r[5]):>8s} {r[6]:7.0f} {r[7]:4d} {r[8]:5d} {r[9]:6.1f}")
reps = np.array([[r[3], r[4]] for r in rows])
print("planes with fewer than 6 repeats of some condition:",
      [(r[0], r[1], r[3]) for r in rows if r[3] < 6])
print()
print("Implied frame rate if each cycle is the stimulus plus a 3 s blank (Methods):")
for m in P:
    isi = np.median(np.concatenate([np.diff(stims[(m, d)][:, 0]) for d in P[m]]))
    dur = pvc10.stats_table(m)[0, 1]
    print(f"  mouse{m}: median onset interval {isi:.0f} frames, cycle {dur + 3:.1f} s -> "
          f"{isi / (dur + 3):.1f} frames/s; at 30 Hz the interval is {isi / 30:.2f} s")

Two departures from the paper show in this table. All six planes of mouse 10 have 4 or 5 repeats of
each condition, and single planes of mice 6 and 8 have 5, against the 6 to 7 repeats per plane of
the Methods; the structs below hold the same numbers of trials per condition, so they are what the
authors analysed. Mouse 10 has a plane at 125 um (cell above), above the 150 um the Methods give as
the shallowest. Both are recorded as findings; the structs' cells of these planes are kept, as
the authors kept them.

### Movies

For every plane, the frame count and frame shape of both channels, the range of the first frame,
and a hash of the first green frame, to look for recordings deposited twice. Only the first frame
of each file is read here.

In [ ]:
t = time.time()
movies = {}
for m, ds in P.items():
    for d in ds:
        for ch in ("green", "red"):
            mv = pvc10.Movie(m, d, ch)
            f0 = mv.frame(0)
            movies[(m, d, ch)] = dict(n=mv.n_frames, shape=f0.shape, dtype=str(mv.frame_dataset(0).dtype),
                                      lo=f0.min(), hi=f0.max(), mean=f0.mean(),
                                      h=hashlib.md5(f0.tobytes()).hexdigest(), img=f0[::4, ::4].copy())
            mv.close()
print(f"read {len(movies)} files in {time.time() - t:.0f} s")
print("frame shapes (rows, columns):", collections.Counter(v["shape"] for v in movies.values()))
print("dtypes:", collections.Counter(v["dtype"] for v in movies.values()))
for ch in ("green", "red"):
    lo = [v["lo"] for k, v in movies.items() if k[2] == ch]
    hi = [v["hi"] for k, v in movies.items() if k[2] == ch]
    mn = [v["mean"] for k, v in movies.items() if k[2] == ch]
    print(f"{ch}: first-frame minimum {min(lo):.0f} to {max(lo):.0f}, maximum {min(hi):.0f} to {max(hi):.0f}, "
          f"mean {min(mn):.1f} to {max(mn):.1f}")
mismatch = [(m, d) for m, ds in P.items() for d in ds if movies[(m, d, 'green')]['n'] != movies[(m, d, 'red')]['n']]
print("planes whose green and red frame counts differ:", mismatch)
short = [(m, d) for m, ds in P.items() for d in ds
         if stims[(m, d)][-1, 0] + np.median(np.diff(stims[(m, d)][:, 0])) > movies[(m, d, 'green')]['n']]
print("planes whose last stimulus cycle runs past the last frame:", short)
nf = [movies[(m, d, 'green')]['n'] for m, ds in P.items() for d in ds]
print("frames per plane:", min(nf), "to", max(nf))
hashes = collections.Counter(v["h"] for k, v in movies.items())
print("first frames identical between files:", [h for h, c in hashes.items() if c > 1])
# similarity of first green frames between planes of the same mouse
for m, ds in P.items():
    imgs = np.array([movies[(m, d, 'green')]['img'].ravel() for d in ds])
    c = np.corrcoef(imgs)
    np.fill_diagonal(c, np.nan)
    print(f"mouse{m}: highest correlation of first green frames between two planes {np.nanmax(c):.3f}")
print("descriptor: frames of 255 by 456 pixels; paper: 256 by 455 pixels; 300 um field")

### What the published files hold per cell

The paper's cells "were identified by hand from structure images based on size, shape, and
brightness" and PV+ cells "by selecting OGB-1-filled neurons with co-localized tdTomato
fluorescence" (Experimental Procedures, Data Imaging Analysis). Every claim needs those cells,
their masks and their PV+/PV- labels. The only masks in the published folder are the example ones
in `live/script/cellmasks.mat`.

In [ ]:
cm = pvc10.load_small_mat(pvc10.LIVE + "script/cellmasks.mat")
for k, v in cm.items():
    print(k, v.shape, v.dtype, "values", np.unique(v).tolist(), "pixels per mask",
          int(v.sum(axis=(0, 1)).min()), "to", int(v.sum(axis=(0, 1)).max()))
print("files in live/ whose names suggest per-cell data (mask, cell, roi, binoc, peak, dsi):",
      [k.split("live/")[1] for k, _ in listing if k.startswith(pvc10.LIVE) and not k.endswith("/")
       and any(w in k.rsplit("/", 1)[1].lower() for w in ("mask", "cell", "roi", "binoc", "peak", "dsi"))])

The bucket listing of `original/`, with what each MATLAB file holds:


In [ ]:
t = time.time()
import fsspec, h5py
for k, s in pvc10.list_bucket(pvc10.DATASET + "/original/"):
    if not k.endswith(".mat"):
        continue
    name = k.rsplit("/", 1)[1]
    if name.startswith("binocmonocinds"):
        d = pvc10.load_small_mat(k)
        print(f"{name:28s} {s:>11d} B  indices: {d['inds'].size}")
        continue
    try:
        fh = fsspec.open(pvc10.BUCKET + k, "rb", block_size=2**20).open()
        h = h5py.File(fh, "r")
    except OSError as e:
        print(f"{name:28s} {s:>11d} B  cannot be opened: {e}")
        continue
    for top in [x for x in h.keys() if not x.startswith("#")]:
        g = h[top]
        fields = list(g.keys())
        deps = collections.Counter(float(h[r][()].ravel()[0]) for r in g["depth"][:, 0])
        print(f"{name:28s} {s:>11d} B  struct {top}: {g[fields[0]].shape[0]} entries, depths {sorted(deps)}")
    h.close()
print("fields:", fields)
print(f"({time.time() - t:.0f} s)")

### The processed structs in `original/to_remove/`

The bucket holds, under the data set's own prefix `pvc-10/`, a folder `original/to_remove/` that
the published file list and checksums do not describe. Its MATLAB structs hold per-cell data of the
kind the descriptor promises ("fluorescence traces, peak responses per condition, ... disparity
selectivity index, preferred relative disparity, the position of the cell"), which the published
folder `live/` lacks.

Decision. These files are read and used for the claims they can test, for mice 8, 9 and 10 (both
populations) and mouse 7 (PV+ only). The basis: they are public, in the archive's own bucket under
the data set's identifier, and each file read is identified by the ETag of the S3 listing, printed
below (for a single-part upload, the MD5 of its content), so a later change is detectable. They are
not part of the data set as published: they are in neither `live/filelist.txt` nor
`live/checksums.md5`, the folder's name says they are to be removed, and nothing documents them.
Every outcome that uses them says so, and rests on the assumption, checked as far as the data allow
below, that their cells are the paper's cells. The checks below are: what one entry is, whether
the per-trial responses come from the published movies, how the stored indices are computed, and
which cells the structs hold.

In [ ]:
tags = pvc10.etags()
print(f"{'file':28s} {'bytes':>11s}  ETag (MD5 of content for single-part uploads)")
for k, (s, e) in sorted(tags.items()):
    if k.endswith(".mat"):
        print(f"{k.rsplit('/', 1)[1]:28s} {s:11d}  {e}")

In [ ]:
t = time.time()
STRUCT_KEYS = [(8, True), (8, False), (9, True), (9, False), (10, True), (10, False), (7, True)]
S = {key: pvc10.load_struct(*key, masks=True) for key in STRUCT_KEYS}
print(f"read {len(S)} structs in {time.time() - t:.0f} s; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
print("mice 5 and 6 have no struct; the three mouse 7 files without the PV suffix are truncated (cell above)")
print()
print("One entry, printed in full for binoc8PV.mat entry 1 (MATLAB orientation):")
s = S[(8, True)]
for f in pvc10.STRUCT_FIELDS:
    v = s[f][0]
    print(f"  {f:14s}", repr(v) if isinstance(v, str) else f"shape {v.shape}, first values {np.round(v.ravel()[:6], 4).tolist()}")
print("  mask: pixels", s["mask_npix"][0], "connected components", s["mask_components"][0],
      "centroid (column, row)", np.round(s["mask_centroid"][0], 2))
print()
print(f"{'struct':12s} {'cells':>5s} {'mask px':>11s} {'1 comp.':>7s} {'|xy - centroid| max':>19s} "
      f"{'date':>11s}  planes (depth: cells)")
for (m, pv), s in S.items():
    xy = np.array([v.ravel() for v in s["xy"]])
    cen = np.array(s["mask_centroid"])
    deps = collections.Counter(float(v.ravel()[0]) for v in s["depth"])
    print(f"mouse{m:<2d} {'PV+' if pv else 'PV-':4s} {s['n']:5d} {min(s['mask_npix']):4d} to {max(s['mask_npix']):3d} "
          f"{sum(c == 1 for c in s['mask_components']):7d} {np.abs(xy - cen).max():19.2f} "
          f"{'/'.join(sorted(set(s['date']))):>11s}  {dict(sorted((int(d), c) for d, c in deps.items()))}")
print("xy range (pixels): column", min(v.ravel()[0] for s in S.values() for v in s["xy"]), "to",
      max(v.ravel()[0] for s in S.values() for v in s["xy"]), "; row",
      min(v.ravel()[1] for s in S.values() for v in s["xy"]), "to", max(v.ravel()[1] for s in S.values() for v in s["xy"]))
print("plane shifts (xyshft) in mice 8 to 10:", {tuple(v.ravel()) for k, s in S.items() if k[0] != 7 for v in s["xyshft"]},
      "; mouse 7:", dict(collections.Counter(tuple(v.ravel()) for v in S[(7, True)]["xyshft"])),
      "; stats.mat shift of mouse 7: (157, -70) um")
print("window length field:", {float(v.ravel()[0]) for s in S.values() for v in s["windowlength"]})

Each entry is one cell: one mask of 11 to 70 pixels, with `xy` its centroid in pixels (column,
row). Below: the trials per condition in `peaks` against the stimulus tables of the published
folder, whether `neuropilpeaks` is one region per plane or per cell, and whether the stored
`standard` and `dsi` follow the Methods' formulas from `peaks`.

In [ ]:
T_all = pvc10.cell_table(S)
ph = np.deg2rad(np.arange(0, 360, 45))
print("trials per condition in peaks against repeats in the published stim file of the same plane:")
for (m, pv), s in S.items():
    agree, total = 0, 0
    for i in range(s["n"]):
        d = int(s["depth"][i].ravel()[0])
        ntr = np.sum(~np.isnan(s["peaks"][i]), 1)
        reps = np.bincount(stims[(m, d)][:, 1], minlength=12)[1:]
        total += 1
        # a trial whose window runs past the movie's end cannot be measured
        agree += np.all((ntr == reps) | (ntr == reps - 1))
    print(f"  mouse{m} {'PV+' if pv else 'PV-'}: {agree} of {total} cells have, for every condition, the stim file's repeats or one fewer")
same_np = []
for (m, pv), s in S.items():
    by_file = collections.defaultdict(list)
    for i in range(s["n"]):
        by_file[s["filename"][i]].append(np.nan_to_num(s["neuropilpeaks"][i]))
    same_np.append(all(all(np.array_equal(v[0], w) for w in v) for v in by_file.values()))
print("neuropilpeaks identical for all cells of a plane, in every struct:", all(same_np))

st_re = (T_all["mu"] - T_all["mu"][:, :1]) / (T_all["se"] + T_all["se"][:, :1])
err_st = np.nanmax(np.abs(st_re[:, 1:] - T_all["standard"][:, 1:]), 1)
print(f"stored 'standard' = (mean stim - mean blank) / (SE stim + SE blank) from peaks within 0.001 for "
      f"{np.sum(err_st < 1e-3)} of {len(err_st)} cells (median difference {np.median(err_st):.4f}, max {err_st.max():.2f})")
for name, (b, c) in {"blank subtracted, as measured": (True, False), "as measured, no blank": (False, False),
                     "blank subtracted, negatives set to zero": (True, True)}.items():
    d_re, p_re = pvc10.dsi_pref(T_all, b, c)
    e = np.abs(d_re - T_all["dsi"])
    ep = np.abs((p_re - T_all["pref"] + 180) % 360 - 180)
    print(f"stored dsi against vector strength of the 8 binocular means, {name}: within 0.001 for "
          f"{np.sum(e < 1e-3)} of {len(e)} cells; preference within 1 deg for {np.sum(ep < 1)}")
print("the stored DSI is the Methods' vector strength of the blank-subtracted responses with negatives set to zero;"
      " the cells where the vector strength with negatives kept differs from it, and where 'standard' differs, by struct:")
d_re, _ = pvc10.dsi_pref(T_all, True, False)
for (m, pv) in S:
    k = (T_all["mouse"] == m) & (T_all["pv"] == pv)
    print(f"  mouse{m} {'PV+' if pv else 'PV-'}: dsi with negatives kept differs for {np.sum(np.abs(d_re[k] - T_all['dsi'][k]) >= 1e-3)} of {k.sum()},"
          f" 'standard' differs for {np.sum(err_st[k] >= 1e-3)}")

Which cells the structs hold. The paper identified 338 PV+ and 3,982 PV- cells in 6 mice, of which
236 PV+ and 2,001 PV- were responsive to binocular stimuli. If the structs held every identified
cell, many would fail the response criterion. The Methods' criterion is applied with the stored
index, with and without the neuropil requirement (cell mean above the neuropil mean + 1.96 SE for
the same condition; see the helper). The `binocmonocinds*` files hold index lists, compared here
with the entry counts and the criterion.

In [ ]:
print(f"{'struct':10s} {'cells':>5s} | without neuropil: {'bino':>4s} {'mono':>4s} {'joint':>5s} {'none':>4s} | "
      f"with neuropil: {'bino':>4s} {'mono':>4s} {'joint':>5s} | binocmonocinds: {'n':>3s} {'max':>3s} {'in joint':>8s} {'in joint+np':>11s}")
for (m, pv) in S:
    k = (T_all["mouse"] == m) & (T_all["pv"] == pv)
    sub = {f: v[k] for f, v in T_all.items()}
    w, n = pvc10.responsive(sub, False), pvc10.responsive(sub, True)
    inds = pvc10.binocmonocinds(m, pv)
    jw = set(np.nonzero(w["joint"])[0] + 1)
    jn = set(np.nonzero(n["joint"])[0] + 1)
    print(f"m{m:<2d} {'PV+' if pv else 'PV-'}   {k.sum():5d} | {'':17s} {w['bino'].sum():4d} {w['mono'].sum():4d} {w['joint'].sum():5d} "
          f"{(~w['bino'] & ~w['mono']).sum():4d} | {'':14s} {n['bino'].sum():4d} {n['mono'].sum():4d} {n['joint'].sum():5d} | "
          f"{'':15s} {len(inds):3d} {inds.max():3d} {len(set(inds) & jw):8d} {len(set(inds) & jn):11d}")
for m in (5, 6, 7):
    for pv in (True, False):
        inds = pvc10.binocmonocinds(m, pv)
        print(f"binocmonocinds{m}{'PV' if pv else ''}: {len(inds)} indices, {inds.min()} to {inds.max()} (no readable struct)")
tot = {pv: sum(len(pvc10.binocmonocinds(m, pv)) for m in range(5, 11)) for pv in (True, False)}
print(f"binocmonocinds totals over 6 mice: PV+ {tot[True]}, PV- {tot[False]}; the paper's monocular-responsive counts: 115 PV+, 944 PV-")
print("struct PV+ cells in mice 7 to 10:", sum(S[(m, True)]["n"] for m in (7, 8, 9, 10)),
      "; PV- cells in mice 8 to 10:", sum(S[(m, False)]["n"] for m in (8, 9, 10)))

Do the structs come from the published movies? On one plane (mouse 8, depth 205 um, struct file
`002_005`), the green movie of `live/` is streamed and the mean raw intensity in each struct mask is
taken per frame. If the structs were made from these files, this should equal the stored `Ftrace`.
The per-trial responses computed from it, as in the clock check below (Fo the 10 frames before
onset, a 0.5 s window centred on the plane's average peak), are then compared with the stored
`peaks`, trial by trial.

In [ ]:
LM, LD = 8, 205
link_masks, link_peaks, link_F, link_lab = [], [], [], []
for pv in (True, False):
    sel = [i for i in range(S[(LM, pv)]["n"]) if int(S[(LM, pv)]["depth"][i].ravel()[0]) == LD]
    link_masks.append(pvc10.struct_masks(LM, pv, sel))
    link_F += [f.ravel() for f in pvc10.struct_field(LM, pv, sel, "Ftrace")]
    link_peaks += [S[(LM, pv)]["peaks"][i] for i in sel]
    link_lab += ["PV+" if pv else "PV-"] * len(sel)
link_masks = np.concatenate(link_masks, axis=2)
mv = pvc10.Movie(LM, LD, "green")
ltr, lsecs = pvc10.mask_traces(mv, link_masks)
mv.close()
raw = ltr + pvc10.DARKNOISE  # mask_traces subtracts DARKNOISE; Ftrace does not
print(f"mouse{LM} depth {LD}: {link_masks.shape[2]} struct cells ({dict(collections.Counter(link_lab))}), "
      f"{ltr.shape[1]} frames streamed in {lsecs:.0f} s; Ftrace lengths {sorted({len(f) for f in link_F})}")
print("largest |mask mean of the published movie - stored Ftrace| per cell:",
      [float(np.round(np.max(np.abs(raw[c] - link_F[c])), 6)) for c in range(len(link_F))])
lst = stims[(LM, LD)]
lx = pvc10.trial_dff(raw, lst, 80)
lpk = int(np.argmax(np.nanmean(lx[:, lst[:, 1] != 1], axis=(0, 1))))
for name, fps in (("30 Hz (paper)", 30.0), ("26.2 Hz (onset intervals)", 26.2)):
    h = int(round(0.5 * fps)) // 2
    lr = np.nanmean(lx[:, :, max(lpk - h, 0):lpk + h + 1], axis=2)
    ours, theirs = [], []
    for c in range(len(link_peaks)):
        for cond in range(1, 12):
            a = lr[c, lst[:, 1] == cond]
            b = link_peaks[c][cond - 1]
            b = b[~np.isnan(b)]
            n = min(len(a), len(b))
            ours += list(a[:n]); theirs += list(b[:n])
    ours, theirs = np.array(ours), np.array(theirs)
    ok = np.isfinite(ours)
    print(f"window {2 * h + 1} frames ({name}) centred on frame {lpk}: trial responses against stored peaks, "
          f"r = {np.corrcoef(ours[ok], theirs[ok])[0, 1]:.3f} over {ok.sum()} trials; "
          f"median |difference| {np.median(np.abs(ours[ok] - theirs[ok])) * 100:.2f}% dF/F")
rng_link = np.random.default_rng(1)
perm = np.array([np.corrcoef(rng_link.permutation(ours[ok]), theirs[ok])[0, 1] for _ in range(200)])
print(f"same with trials permuted: r 95% range {np.percentile(perm, 2.5):.3f} to {np.percentile(perm, 97.5):.3f}")

### The clock of the stimulus table, on the example plane

The descriptor's example plane (mouse5, depth 260 um) is streamed whole, the mean intensity in each
of the 50 example masks is taken per frame (minus the offset of -160 that the data set's
`examplecode.m` subtracts), and dF/F per trial is computed with Fo the 10 frames before onset (a
standard choice; see the helper). If the stimulus table shares the movie's clock, stimulus trials
should differ from blank trials in a window after onset, and the difference should fall when the
onsets are moved or the condition labels shuffled. This is a check of the clock only. The window
is centred on the peak of the average over all masks (the plane), where the paper's plain reading
is per cell; a per-mask window is shown as a variant. Either way the window is placed on the same
trials it then measures, so the stimulus-minus-blank value is biased upward; the shuffled labels
and the onset shifts, which use the same window, are the comparison. The data set gives no frame
rate: the window is 15 frames (0.5 s at the paper's 30 Hz), and 13 frames (0.5 s at the 26.2
frames/s the onset intervals imply for this mouse) is shown; the conflict between the two rates is
not resolved here and is recorded as a finding. This checks one plane of 58; the others are not
checked, except mouse 8 depth 205 through the structs above.

In [ ]:
M, D = 5, 260
masks = cm["cellmasks"]
mv = pvc10.Movie(M, D, "green")
traces, secs = pvc10.mask_traces(mv, masks)
mv.close()
stim = stims[(M, D)]
size_gb = dict(live_data)[pvc10.plane_path(M, D, "green")] / 1e9
print(f"streamed {traces.shape[1]} frames ({size_gb:.2f} GB) in {secs:.0f} s, "
      f"{size_gb * 1e3 / secs:.1f} MB/s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
print("raw trace range", round(traces.min(), 1), "to", round(traces.max(), 1))

N_POST = 118  # one onset interval of this plane (median 118 frames)
x = pvc10.trial_dff(traces, stim, N_POST)
lab = stim[:, 1]
avg_stim = np.nanmean(x[:, lab != 1], axis=(0, 1)) * 100
avg_blank = np.nanmean(x[:, lab == 1], axis=(0, 1)) * 100
peak = int(np.argmax(avg_stim))
print(f"mean dF/F over cells and stimulus trials peaks at frame {peak} after onset, {avg_stim[peak]:.2f}%; "
      f"blank trials at that frame {avg_blank[peak]:.2f}%")
print("mean dF/F (%) every 6 frames, stimulus trials:", np.round(avg_stim[::6], 2).tolist())
print("mean dF/F (%) every 6 frames, blank trials:   ", np.round(avg_blank[::6], 2).tolist())

# the paper's 0.5 s window, centred on the average peak (Methods); 0.5 s is 15 frames at 30 Hz
# (13 frames at the 26.2 frames/s the onset intervals imply; shown as a variant below)
half = 7
win = slice(max(peak - half, 0), peak + half + 1)


def contrast(shift=0, labels=None):
    xx = pvc10.trial_dff(traces, stim, N_POST, shift)
    r = np.nanmean(xx[:, :, win], axis=2)
    ll = lab if labels is None else labels
    return (np.nanmean(r[:, ll != 1]) - np.nanmean(r[:, ll == 1])) * 100, r


c0, R = contrast()
rng = np.random.default_rng(20150630)
null = np.array([contrast(0, rng.permutation(lab))[0] for _ in range(200)])
print(f"stimulus minus blank in the window (frames {win.start} to {win.stop - 1}): {c0:.2f}% dF/F; "
      f"with shuffled labels 95% range {np.percentile(null, 2.5):.2f} to {np.percentile(null, 97.5):.2f}, "
      f"p = {(np.sum(np.abs(null) >= abs(c0)) + 1) / (len(null) + 1):.3f}")
shifts = [-118, -90, -60, -30, 0, 30, 60, 90]
cs = [contrast(s)[0] for s in shifts]
for s, c in zip(shifts, cs):
    print(f"  onsets moved by {s:+4d} frames: stimulus minus blank {c:6.2f}%")

for nm, hh in (("13 frames (0.5 s at 26.2 Hz)", 6),):
    w2 = slice(max(peak - hh, 0), peak + hh + 1)
    r2 = np.nanmean(x[:, :, w2], axis=2)
    print(f"window of {nm}: stimulus minus blank {(np.nanmean(r2[:, lab != 1]) - np.nanmean(r2[:, lab == 1])) * 100:.2f}%")
pk_cell = np.argmax(np.nanmean(x[:, lab != 1], axis=1), axis=1)
rc = np.array([np.nanmean(x[c, :, max(p - half, 0):p + half + 1], axis=1) for c, p in enumerate(pk_cell)])
print(f"window centred on each mask's own peak (frames {pk_cell.min()} to {pk_cell.max()}, median {np.median(pk_cell):.0f}): "
      f"stimulus minus blank {(np.nanmean(rc[:, lab != 1]) - np.nanmean(rc[:, lab == 1])) * 100:.2f}% (circular: the window is placed on the same trials)")

# responsiveness index of the Methods per mask, without the neuropil requirement
n = np.array([np.sum(lab == k) for k in range(1, 12)])
mb, sb = R[:, lab == 1].mean(1), R[:, lab == 1].std(1, ddof=1) / np.sqrt(n[0])
idx = np.stack([(R[:, lab == k].mean(1) - mb) / (R[:, lab == k].std(1, ddof=1) / np.sqrt(n[k - 1]) + sb)
                for k in range(2, 12)], 1)
print("example masks with index >= 1 for some binocular condition:", int((idx[:, :8] >= 1).any(1).sum()),
      "of 50; for some monocular condition:", int((idx[:, 8:] >= 1).any(1).sum()),
      "(the descriptor calls them a mixture of responsive and non-responsive cells)")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(avg_stim, label="stimulus trials")
ax[0].plot(avg_blank, label="blank trials")
ax[0].axvspan(win.start, win.stop - 1, color="0.9")
ax[0].set_xlabel("frames after onset"); ax[0].set_ylabel("mean dF/F (%)"); ax[0].legend()
ax[1].plot(shifts, cs, "o-")
ax[1].axhspan(np.percentile(null, 2.5), np.percentile(null, 97.5), color="0.9")
ax[1].set_xlabel("onset shift (frames)"); ax[1].set_ylabel("stimulus minus blank (%)")
plt.tight_layout(); plt.show()
print(f"plotted ranges: left {min(avg_stim.min(), avg_blank.min()):.2f} to {max(avg_stim.max(), avg_blank.max()):.2f}%; "
      f"right {min(cs):.2f} to {max(cs):.2f}%; nothing outside the axes")

### Red channel of the example masks

The descriptor says the red channel "can be used to identify if the cell is parvalbumin positive or
not", but gives no labels. The mean red intensity in each example mask over the first 160 frames
(the number `examplecode.m` averages) shows what a reader would have to threshold, with no rule from
the paper or the data set to do it.

In [ ]:
mv = pvc10.Movie(M, D, "red")
red, rsecs = pvc10.mask_traces(mv, masks, range(160))
red_img = np.mean([mv.frame(i) for i in range(0, 160, 8)], axis=0)
mv.close()
r = red.mean(1)
print(f"red mean in the 50 masks: {r.min():.1f} to {r.max():.1f} (median {np.median(r):.1f}); "
      f"whole-frame median {np.median(red_img) - pvc10.DARKNOISE:.1f}")
print("masks sorted by red intensity:", np.round(np.sort(r), 0).astype(int).tolist())

### Size of a full re-analysis

Re-running the paper's analysis would need the cells identified anew in all 58 planes (by hand, in
the paper) and both channels streamed. The rate measured above gives the time to stream the movies
alone.

In [ ]:
movie_bytes = sum(s for k, s in live_data if k.endswith("green.mat") or k.endswith("red.mat"))
rate = size_gb * 1e9 / secs
print(f"{sum(1 for k, _ in live_data if k.endswith(('green.mat', 'red.mat')))} movies, {movie_bytes / 1e9:.0f} GB; "
      f"at {rate / 1e6:.1f} MB/s, {movie_bytes / rate / 3600:.1f} h to stream once, against a 1 h notebook limit")
print(f"notebook time so far {(time.time() - T0) / 60:.1f} min")

## Claims

Blocked without analysis: fig1e-pv-tdtomato-colocalization (no histology), the three CaMKIIa claims
(no CaMKIIa animals), cell-counts-responsive (the structs hold a selection of the cells, see the
data check, and the published folder holds no masks), figs3-pv-distinct-from-neuropil and
figs5-neuropil-no-relation (no neuropil patches; the structs hold one neuropil region per plane,
shown above, and no non-responsive PV+ cells). The other claims are tested on the structs of mice
8, 9 and 10, the three animals with both populations, against six in the paper.

Readings used throughout, each from the Methods (Data Imaging Analysis) or the claims file, with
the choices this attempt makes named as such:

- Visually responsive (`resp`). `L`, the main reading: the cells in the `binocmonocinds` lists. This
  is an assumption. The lists are undocumented, but their name reads as cells responsive to
  binocular and monocular stimuli, which is the Methods' joint criterion; 235 of their 270 cells
  in the structs (77 of 81 PV+, 158 of 189 PV-) pass the joint criterion on the stored index (data
  check); they hold fewer cells than that criterion alone gives, as the Methods' further neuropil
  test and check "by eye" would remove; and their totals over the six mice, 109 PV+ and 769 PV-,
  are 5% and 19% below the paper's monocular-responsive counts (115 PV+, 944 PV-). `N`: the Methods' criterion applied here, at least one monocular and
  one binocular stimulus with index >= 1, each also above the plane's neuropil response (mean +
  1.96 SE, a reading of "> 95% confidence interval"). `W`: the same without the neuropil test.
- L decides every outcome; where a criterion says "under one reading", that ranges over the claims
  file's readings within L. The cell set is not chosen claim by claim. L is the set whose counts
  are closest to the paper's; N, built here with a neuropil test the Methods do not define, keeps
  about half as many cells (in the Figure 3 pool 41 PV+ and 92 PV- joint, 47 and 103 monocular,
  against L's 81 and 189). N and W are shown for every claim as the sensitivity of each outcome to
  this choice. Under L the claims file's cell readings (joint, and monocular-responsive) give the
  same cells, and the lists' totals are nearest the counts of the monocular-responsive reading.
- Denominator (`den`, claims file): `joint` (cells passing the criterion) or `mono` (cells with a
  monocular stimulus passing it; for disparity, `bino`).
- Responses for ODI: the mean dF/F minus the blank mean, negatives set to zero (main; the stored
  DSI is computed on blank-subtracted responses, shown above). Variants: as measured (no blank
  subtraction), and blank subtracted without setting negatives to zero.
- DSI and preference: the stored values, which are the vector strength of the blank-subtracted
  responses with negatives set to zero (data check). Variant: blank subtracted, negatives kept
  (claims file, "as measured").
- Cells for the population comparisons of Figure 3: PV+ cells of mice 7 to 10 and PV- cells of
  mice 8 to 10 (main, every cell the structs hold, pooled as the paper pools cells across
  animals); mice 8 to 10 only as a variant. Per-animal and local analyses use mice 8 to 10.
- Local analysis: open readings (1) to (9) of the claims file; main reading (2a), (3a), (6)
  unweighted, (7a) 300 um along each axis, (8a), and both (1a) and (1b), (5) one- and two-sided,
  (9a) and (9b) shown. 1000 bootstrap resamples and 1000 shuffles (standard choice, reading 4).

In [ ]:
from scipy.stats import mannwhitneyu, pearsonr
T = pvc10.cell_table({k: v for k, v in S.items() if k[0] in (8, 9, 10)})
T7 = pvc10.cell_table(S)  # adds mouse 7 PV+ cells, used only as a variant
L_SET = {(m, pv): set(pvc10.binocmonocinds(m, pv)) for m in (7, 8, 9, 10) for pv in (True, False)}


def resp_sets(TT, resp):
    """Boolean selections per denominator for a responsiveness reading N, W or L."""
    if resp == "L":
        lst = np.array([i in L_SET[(m, pv)] for m, pv, i in zip(TT["mouse"], TT["pv"], TT["idx"])])
        return dict(joint=lst, mono=lst, bino=lst)
    return pvc10.responsive(TT, neuropil=(resp == "N"))


POOLS = {"mice 7-10 PV+, 8-10 PV-": T7, "mice 8-10": T}
ODI_READINGS = {"blank-subtracted, negatives to 0": (True, True), "as measured, negatives to 0": (False, True),
                "blank-subtracted, as measured": (True, False)}
print("cells in mice 8 to 10:", {f"{'PV+' if pv else 'PV-'}": int(np.sum(T["pv"] == pv)) for pv in (True, False)})
for resp in "LNW":
    R = resp_sets(T, resp)
    print(f"resp {resp}: " + "; ".join(f"{den} PV+ {np.sum(R[den] & T['pv'])} PV- {np.sum(R[den] & ~T['pv'])}"
                                         for den in ("joint", "mono", "bino")))

### fig3abc-pv-more-binocular-fraction

> A larger proportion of PV+ cells than PV- cells respond to both eyes (64% against 45% with
> -0.5 < ODI < 0.5).

Expected 64 +/- 6.4% (PV+) and 45 +/- 4.5% (PV-); holds when both are within tolerance under one
denominator reading. The same table gives the median ODI (fig3c-contralateral-bias) and MI
(fig3-pv-lower-monocularity).

In [ ]:
def fig3_row(TT, sel, o):
    out = {}
    for pv in (True, False):
        x = o[sel & (TT["pv"] == pv) & np.isfinite(o)]
        out[pv] = x
    a, b = np.abs(out[True]), np.abs(out[False])
    p = mannwhitneyu(a, b, alternative="two-sided").pvalue
    return out, p


fig3 = {}
for pool, TT in POOLS.items():
    print(pool)
    print(f"{'resp':4s} {'den':5s} {'responses':34s} {'nPV+':>4s} {'nPV-':>4s} {'binoc PV+':>9s} {'PV-':>5s} "
          f"{'medODI+':>7s} {'medODI-':>7s} {'MI+ med':>7s} {'mean':>6s} {'SD':>5s} {'MI- med':>7s} {'mean':>6s} {'SD':>5s} {'p MW':>7s}")
    for resp in "LNW":
        R = resp_sets(TT, resp)
        for den in (("joint", "mono") if resp != "L" else ("joint",)):
            for oname, (bl, cl) in ODI_READINGS.items():
                o = pvc10.odi(TT, bl, cl)
                out, p = fig3_row(TT, R[den], o)
                fig3[(pool, resp, den, oname)] = (out, p)
                fr = {pv: 100 * np.mean(np.abs(out[pv]) < 0.5) for pv in out}
                mi = {pv: np.abs(out[pv]) for pv in out}
                print(f"{resp:4s} {den:5s} {oname:34s} {len(out[True]):4d} {len(out[False]):4d} {fr[True]:9.1f} {fr[False]:5.1f} "
                      f"{np.median(out[True]):7.3f} {np.median(out[False]):7.3f} {np.median(mi[True]):7.3f} {mi[True].mean():6.3f} "
                      f"{mi[True].std(ddof=1):5.2f} {np.median(mi[False]):7.3f} {mi[False].mean():6.3f} {mi[False].std(ddof=1):5.2f} {p:7.4f}")
    print()
print("per animal, main reading (L, main responses): n, binocular %, median ODI")
o = pvc10.odi(T7)
R = resp_sets(T7, "L")
for m in (7, 8, 9, 10):
    for pv in (True, False):
        x = o[R["joint"] & (T7["mouse"] == m) & (T7["pv"] == pv) & np.isfinite(o)]
        print(f"  mouse{m} {'PV+' if pv else 'PV-'}: n {len(x)}, {100 * np.mean(np.abs(x) < 0.5) if len(x) else np.nan:.0f}%, "
              f"median ODI {np.median(x) if len(x) else np.nan:.2f}")
o_main = fig3[("mice 7-10 PV+, 8-10 PV-", "L", "joint", "blank-subtracted, negatives to 0")][0]
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for pv, c in ((False, "tab:blue"), (True, "tab:red")):
    ax[0].hist(o_main[pv], bins=np.linspace(-1, 1, 21), histtype="step", density=True, color=c, label="PV+" if pv else "PV-")
    ax[1].plot(np.sort(o_main[pv]), np.linspace(0, 1, len(o_main[pv])), color=c)
ax[0].set_xlabel("ODI"); ax[0].legend(); ax[1].set_xlabel("ODI"); ax[1].set_ylabel("cumulative fraction")
plt.tight_layout(); plt.show()
print("plotted: ODI of the main reading (L, mice 7-10 PV+, 8-10 PV-), all values in [-1, 1]")

**Result: reproduced, resting on the assumption that the `binocmonocinds` lists are the paper's visually responsive cells.** In the main reading (L, PV+ cells of mice 7 to 10 and PV- cells of mice 8 to 10, blank-subtracted responses with negatives set to zero), 60.5% of 81 PV+ cells and 45.5% of 189 PV- cells have -0.5 < ODI < 0.5, both within tolerance (57.6 to 70.4%, 40.5 to 49.5%). With mice 8 to 10 only, 57.7% and 45.5%. The reading N gives 58.5% and 44.6% (joint) and 61.7% and 42.7% (mono), also within. The reading W puts the PV- fraction outside (51.3%, 50.6%), and responses taken without the blank subtracted put the PV+ fraction outside under every cell set (35.8% to 50.0%). The PV+ counts per animal are small (3 to 13 in mice 8 to 10).

### fig3c-contralateral-bias

> Both PV+ and PV- populations are biased toward the contralateral eye.

Reproduced when the median ODI is below 0 in both populations (table above, columns medODI).

**Result: reproduced.** The median ODI is below 0 for both populations in every reading and cell pool in the table: -0.341 (PV+) and -0.442 (PV-) in the main reading, and from -0.630 to -0.013 (PV+) and -0.468 to -0.388 (PV-) across all rows.

### fig3-pv-lower-monocularity

> PV+ interneurons are significantly more binocular than PV- neurons (MI median 0.36, mean 0.41
> +/- 0.29 SD, against median 0.57, mean 0.58 +/- 0.32 SD; p = 0.01, Mann-Whitney).

Expected medians and means within 10%, and a two-sided Mann-Whitney p <= 0.01 with PV+ lower
(table above, columns MI and p MW).

**Result: partially reproduced, resting on the assumption about the `binocmonocinds` lists.** In the main reading PV+ cells are more binocular than PV- cells (two-sided Mann-Whitney p = 0.0019, 81 against 189 cells), the PV+ median MI is 0.370 (tolerance 0.324 to 0.396), the PV- median 0.524 (0.513 to 0.627) and mean 0.590 (0.522 to 0.638), but the PV+ mean is 0.452, just above its tolerance (0.369 to 0.451); the criterion's partial case (test holds, a value outside). The reading N with monocular-responsive cells gives all four values within tolerance and p = 0.0092 (47 PV+, 103 PV-). Within mice 8 to 10 only, the main reading gives p = 0.092 (26 PV+ cells): the significance in the main pool depends on the 55 PV+ cells of mouse 7, which has no PV- struct. The outcome is judged on L, as every claim is (see the readings above). N with monocular-responsive cells would meet the reproduced case, on 47 PV+ and 103 PV- cells against L's 81 and 189, with a neuropil test that is this attempt's reading of an undefined one.

### mi-per-animal

> The monocularity difference holds animal by animal; across the 6 animals the mean PV- MI is
> 0.58 +/- 0.04 SD and the mean PV+ MI 0.43 +/- 0.08 SD.

Three animals here.

In [ ]:
for resp in "LNW":
    R = resp_sets(T, resp)
    for den in (("joint", "mono") if resp != "L" else ("joint",)):
        o = pvc10.odi(T)
        per = {pv: [] for pv in (True, False)}
        txt = []
        for m in (8, 9, 10):
            v = {}
            for pv in (True, False):
                x = np.abs(o[R[den] & (T["mouse"] == m) & (T["pv"] == pv) & np.isfinite(o)])
                v[pv] = (x.mean() if len(x) else np.nan, len(x))
                per[pv].append(v[pv][0])
            txt.append(f"m{m} PV+ {v[True][0]:.2f} (n {v[True][1]}) PV- {v[False][0]:.2f} (n {v[False][1]})")
        a, b = np.array(per[True]), np.array(per[False])
        print(f"{resp} {den:5s}: mean over animals PV- {np.nanmean(b):.2f} +/- {np.nanstd(b, ddof=1):.2f} SD, "
              f"PV+ {np.nanmean(a):.2f} +/- {np.nanstd(a, ddof=1):.2f} SD; PV+ below PV- in {np.sum(a < b)} of 3 | " + "; ".join(txt))

**Result: reproduced on 3 of the paper's 6 animals, resting on the assumption about the `binocmonocinds` lists.** In the main reading the mean over mice 8 to 10 of the per-animal mean MI is 0.59 +/- 0.01 SD for PV- (tolerance 0.522 to 0.638) and 0.44 +/- 0.13 SD for PV+ (0.387 to 0.473), with PV+ below PV- in 3 of 3 animals (in mouse 9 by 0.59 against 0.59, with 13 PV+ cells; mouse 10 has 3). The reading N gives 0.56 and 0.45 (joint) or 0.59 and 0.43 (mono), within; W gives 0.52 and 0.38, outside.

### fig3def-pv-weaker-disparity-selectivity

> PV+ interneurons have weaker disparity selectivity than PV- neurons (DSI median 0.09, mean 0.12
> +/- 0.10 SD, against median 0.26, mean 0.26 +/- 0.17 SD; p = 0.0001, Mann-Whitney).

Readings (claims file): cells (a) joint, (b) binocular-responsive; negative responses (a) as
measured, (b) set to zero (the stored DSI, data check).

In [ ]:
for pool, TT in POOLS.items():
    d_meas, _ = pvc10.dsi_pref(TT, True, False)
    DSI_READINGS = {"stored (negatives to 0)": TT["dsi"], "as measured": d_meas}
    print(pool)
    print(f"{'resp':4s} {'den':5s} {'DSI':24s} {'nPV+':>4s} {'nPV-':>4s} {'PV- med':>7s} {'mean':>5s} {'SD':>5s} "
          f"{'PV+ med':>7s} {'mean':>5s} {'SD':>5s} {'p MW':>9s}")
    for resp in "LNW":
        R = resp_sets(TT, resp)
        for den in (("joint", "bino") if resp != "L" else ("joint",)):
            for dn, dv in DSI_READINGS.items():
                a = dv[R[den] & TT["pv"] & np.isfinite(dv)]
                b = dv[R[den] & ~TT["pv"] & np.isfinite(dv)]
                p = mannwhitneyu(a, b, alternative="two-sided").pvalue
                print(f"{resp:4s} {den:5s} {dn:24s} {len(a):4d} {len(b):4d} {np.median(b):7.3f} {b.mean():5.3f} {b.std(ddof=1):5.3f} "
                      f"{np.median(a):7.3f} {a.mean():5.3f} {a.std(ddof=1):5.3f} {p:9.2e}")
    print("range of the DSI values:", round(np.nanmin(TT["dsi"]), 3), "to", round(np.nanmax(TT["dsi"]), 3),
          "(stored);", round(np.nanmin(d_meas), 3), "to", round(np.nanmax(d_meas), 3), "(as measured; above 1 when negative responses enter)")
    print()

**Result: partially reproduced.** PV+ cells have much weaker disparity selectivity in every reading (two-sided Mann-Whitney p from 7.2e-17 to 2.4e-04 in the main pool, PV+ lower), but no reading puts all four values within tolerance. In the main reading the PV- median is 0.230 (tolerance 0.234 to 0.286) and mean 0.255 (0.234 to 0.286), the PV+ median 0.088 (0.081 to 0.099) and mean 0.101 (0.108 to 0.132). The N and W readings with binocular-responsive cells give PV- medians of 0.260 and 0.243 and a W PV- mean of 0.280 within tolerance, but an N PV- mean of 0.295 and PV+ medians and means of 0.103 to 0.144, outside. Within mice 8 to 10 only, the reading N with joint cells (12 PV+) gives p = 0.33. In the main reading, DSI taken as measured (negatives kept) changes only the PV- mean (to 0.747, as some values exceed 1); under N and W it also moves some PV- medians and PV+ means (for example W with binocular-responsive cells, PV+ mean 0.144 to 0.154).

### mi-dsi-relation

> Monocularity and disparity selectivity are unrelated in PV+ neurons (mean slope 0.04 +/- 0.02 SE,
> bootstrapped PCA); in PV- neurons there is a weak positive relation (mean slope 0.16 +/- 0.03 SE)
> that accounts for little variance (r2 = 0.08).

Readings (claims file): (a) DSI on MI, (b) MI on DSI; cells with an MI among the DSI cells.

In [ ]:
rng = np.random.default_rng(301)
for pool, TT in POOLS.items():
    print(pool)
    print(f"{'resp':4s} {'den':5s} {'axis':10s} {'PV+ n':>5s} {'slope':>6s} {'SE':>6s} {'PV- n':>5s} {'slope':>6s} {'SE':>6s} {'PV- r2':>6s} {'PV+ r2':>6s}")
    o = pvc10.odi(TT)
    for resp in "LNW":
        R = resp_sets(TT, resp)
        for den in (("joint", "bino") if resp != "L" else ("joint",)):
            for axis in ("DSI on MI", "MI on DSI"):
                row = []
                r2 = {}
                for pv in (True, False):
                    k = R[den] & (TT["pv"] == pv) & np.isfinite(o) & np.isfinite(TT["dsi"])
                    mi, d = np.abs(o[k]), TT["dsi"][k]
                    x, y = (mi, d) if axis == "DSI on MI" else (d, mi)
                    b = pvc10.boot_slopes(x, y, rng)
                    b = b[np.isfinite(b)]
                    row.append((k.sum(), b.mean(), b.std(ddof=1)))
                    r2[pv] = pearsonr(mi, d)[0] ** 2
                print(f"{resp:4s} {den:5s} {axis:10s} {row[0][0]:5d} {row[0][1]:6.3f} {row[0][2]:6.3f} {row[1][0]:5d} "
                      f"{row[1][1]:6.3f} {row[1][2]:6.3f} {r2[False]:6.4f} {r2[True]:6.4f}")
    print()

**Result: not reproduced, resting on the assumption about the `binocmonocinds` lists.** In the main reading, with DSI on MI (reading a), the PV- slope is 0.178 +/- 0.050 (tolerance 0.13 to 0.19) and the PV- r2 0.0723 (0.072 to 0.088), but the PV+ slope is 0.013 +/- 0.025, below its tolerance of 0.02 to 0.06; the criterion's partial case needs the PV+ slope within tolerance. PV+ MI and DSI are unrelated here too (r2 0.0033). The reading N with joint cells gives a PV+ slope of 0.059 and a PV- slope of 0.151 within tolerance but r2 0.054, outside (partial); W with joint cells gives 0.044 and 0.080 (partial). Reading (b), MI on DSI, gives unstable slopes (SE above 1) in every case. The outcome is judged on L, as every claim is (see the readings above); the N and W cases meeting the partial case rest on other cell sets (41 PV+ and 92 PV- cells for N joint, against L's 81 and 189).

### pv-larger-dff-low-dsi

> Among cells with comparable weak disparity selectivity (DSI < 0.1), PV+ interneurons have larger
> responses than PV- neurons (median dF/F over all binocular stimuli 6.2% against 4.3%; to the best
> binocular stimulus 8.7% against 6.6%; both p < 0.001, Mann-Whitney).

Responses are the mean dF/F per condition as stored (`peaks`, not blank subtracted). Readings
(claims file): "all binocular" (a) each cell's mean over the eight conditions, (b) all
cell-by-condition responses pooled; "best" each cell's largest binocular mean. DSI is the stored
value; cells as in fig3def (joint, and binocular-responsive).

In [ ]:
for pool, TT in POOLS.items():
    print(pool)
    print(f"{'resp':4s} {'den':5s} {'nPV+':>4s} {'nPV-':>4s} | (a) PV+ {'med%':>5s} PV- {'med%':>5s} {'p':>8s} | "
          f"(b) PV+ {'med%':>5s} PV- {'med%':>5s} {'p':>8s} | best PV+ {'med%':>5s} PV- {'med%':>5s} {'p':>8s}")
    for resp in "LNW":
        R = resp_sets(TT, resp)
        for den in (("joint", "bino") if resp != "L" else ("joint",)):
            g = {}
            for pv in (True, False):
                k = R[den] & (TT["pv"] == pv) & (TT["dsi"] < 0.1)
                bm = TT["mu"][k, 1:9] * 100
                g[pv] = (bm.mean(1), bm.ravel(), bm.max(1), k.sum())
            pa = mannwhitneyu(g[True][0], g[False][0], alternative="two-sided").pvalue
            pb = mannwhitneyu(g[True][1], g[False][1], alternative="two-sided").pvalue
            pc = mannwhitneyu(g[True][2], g[False][2], alternative="two-sided").pvalue
            print(f"{resp:4s} {den:5s} {g[True][3]:4d} {g[False][3]:4d} | {np.median(g[True][0]):12.2f} {np.median(g[False][0]):9.2f} {pa:8.1e} | "
                  f"{np.median(g[True][1]):12.2f} {np.median(g[False][1]):9.2f} {pb:8.1e} | {np.median(g[True][2]):13.2f} "
                  f"{np.median(g[False][2]):9.2f} {pc:8.1e}")
    print()

**Result: not reproduced.** No reading gives both tests at p < 0.001 with PV+ larger, as the criterion requires. In the main pool PV+ responses are larger in every reading (main reading: medians 6.94% against 5.02% over all binocular conditions per cell, p = 0.0089; 6.91% against 5.22% pooled, p = 1.7e-09; best condition 9.23% against 7.47%, p = 0.017; 51 PV+ and 32 PV- cells with DSI < 0.1), and the medians are mostly above the paper's (PV- 3.92% to 5.87% against 4.3%). Within mice 8 to 10 only, no reading gives a difference at p < 0.001 (p from 0.021 to 0.98; the lowest, N with binocular-responsive cells and responses pooled, has PV- larger), and PV+ and PV- medians are within 0.7 percentage points of each other, so the difference in the main pool comes from the PV+ cells of mouse 7.

### Local network analysis: positions

Positions in um: the mask centroid scaled by 300 um per 455 columns and per 256 rows (reading 7a)
or 300/455 um per pixel on both axes (7b), plus depth. Individual cells must lie 50 um or more from
the image edges. Populations come from the same mouse, the 50 um sphere and the shells to 150 um,
at least 3 cells, the individual left out.

In [ ]:
pos = {sq: pvc10.positions_um(T, sq) for sq in (False, True)}
edge = {sq: pvc10.edge_ok(T, sq) for sq in (False, True)}
for sq in (False, True):
    print(f"scale {'7b square pixels' if sq else '7a 300 um per axis'}: x {pos[sq][:, 0].min():.0f} to {pos[sq][:, 0].max():.0f} um, "
          f"y {pos[sq][:, 1].min():.0f} to {pos[sq][:, 1].max():.0f} um; cells 50 um from the edges: "
          f"PV+ {np.sum(edge[sq] & T['pv'])} of {np.sum(T['pv'])}, PV- {np.sum(edge[sq] & ~T['pv'])} of {np.sum(~T['pv'])}")
print("depths per mouse:", {m: sorted(set(T["depth"][T["mouse"] == m].astype(int))) for m in (8, 9, 10)})

OD_CONFIGS = {"main (L, 7a, 3a)": dict(resp="L", den="joint", sq=False, drop_all=False),
              "N (neuropil test), joint": dict(resp="N", den="joint", sq=False, drop_all=False),
              "N, mono (reading 2b)": dict(resp="N", den="mono", sq=False, drop_all=False),
              "W (no neuropil test), joint": dict(resp="W", den="joint", sq=False, drop_all=False),
              "W, mono": dict(resp="W", den="mono", sq=False, drop_all=False),
              "L, square pixels (7b)": dict(resp="L", den="joint", sq=True, drop_all=False),
              "L, all radii (3b)": dict(resp="L", den="joint", sq=False, drop_all=True)}


def run_od(ind_pv, cfg, axis, seed, ind_filter=None, pop_filter=None, plane="all"):
    R = resp_sets(T, cfg["resp"])
    o = pvc10.odi(T)
    base = R[cfg["den"]] & np.isfinite(o)
    ind = base & (T["pv"] == ind_pv) & edge[cfg["sq"]]
    pop = base & ~T["pv"]
    if ind_filter is not None:
        ind &= ind_filter(o)
    if pop_filter is not None:
        pop &= pop_filter(o)
    return pvc10.od_local(T, np.nonzero(ind)[0], np.nonzero(pop)[0], o, pos[cfg["sq"]],
                          np.random.default_rng(seed), axis=axis, drop_all=cfg["drop_all"], plane=plane)


def show(res, label):
    cells = []
    for r in res:
        if "slope" not in r:
            cells.append(f"{r['radius']}: n {r['n']}")
            continue
        cells.append(f"{r['radius']}: n {r['n']} {r['slope']:.2f}+/-{r['se']:.2f} sh {r['shuf_mean']:.2f} "
                     f"p1 {r['p_one']:.3f} p2 {r['p_two']:.3f}")
    print(f"{label:40s} | " + " | ".join(cells))

### fig4ag-salt-and-pepper

> Mouse V1 has no functional organization for ocular dominance or disparity preference; response
> similarity is unrelated to distance between cells (MI, bootstrapped PCA slope 0.01, p = 1;
> disparity, circular-linear correlation 0, p = 1, cells with DSI > 0.1).

A claim of absence at p < 0.05. Pairs of visually responsive cells (PV+ and PV-) within a mouse,
3D distance in um. MI reading (a): |MI difference| against distance, major-axis slope, p (i) from
the bootstrap (two-sided, against zero), (ii) against slopes with positions shuffled within the
mouse. Disparity: preference difference against distance, CircStat circ_corrcl.

In [ ]:
rng = np.random.default_rng(401)


def pairs(sel, sq, pos_override=None):
    P = pos[sq] if pos_override is None else pos_override
    I, J = [], []
    for m in (8, 9, 10):
        idx = np.nonzero(sel & (T["mouse"] == m))[0]
        a, b = np.triu_indices(len(idx), 1)
        I.append(idx[a]); J.append(idx[b])
    I, J = np.concatenate(I), np.concatenate(J)
    return I, J, np.linalg.norm(P[I] - P[J], axis=1)


for name, cfg in [("main (L, 7a)", ("L", "joint", False)), ("N", ("N", "joint", False)),
                  ("W", ("W", "joint", False)), ("L, square pixels (7b)", ("L", "joint", True))]:
    resp, den, sq = cfg
    R = resp_sets(T, resp)
    o = pvc10.odi(T)
    sel = R[den] & np.isfinite(o)
    I, J, dist = pairs(sel, sq)
    dmi = np.abs(np.abs(o[I]) - np.abs(o[J]))
    b = pvc10.boot_slopes(dist, dmi, rng)
    b = b[np.isfinite(b)]
    p_boot = min(1.0, 2 * min(np.mean(b <= 0), np.mean(b >= 0)))
    sh = []
    for _ in range(1000):
        P = pos[sq].copy()
        for m in (8, 9, 10):
            k = np.nonzero(sel & (T["mouse"] == m))[0]
            P[k] = P[rng.permutation(k)]
        _, _, ds = pairs(sel, sq, P)
        sh.append(pvc10.ma_slope(ds, dmi))
    sh = np.array(sh)
    m_ = b.mean()
    p_sh = (np.sum(np.abs(sh - sh.mean()) >= abs(m_ - sh.mean())) + 1) / (len(sh) + 1)
    print(f"MI {name:20s}: {len(I)} pairs ({sel.sum()} cells), slope {m_:.2e} per um ({m_ * 100:.4f} per 100 um) +/- {b.std(ddof=1):.1e}; "
          f"Pearson r {np.corrcoef(dist, dmi)[0, 1]:.3f}; p (i) bootstrap {p_boot:.3f}, (ii) shuffle {p_sh:.3f}")
    ds_sel = sel & (T["dsi"] > 0.1)
    I, J, dist = pairs(ds_sel, sq)
    dpref = np.deg2rad(T["pref"][I] - T["pref"][J])
    rho, p = pvc10.circ_corrcl(dpref, dist)
    print(f"   disparity: {len(I)} pairs ({ds_sel.sum()} cells with DSI > 0.1), circ_corrcl rho {rho:.3f}, p {p:.3f}")

**Result: reproduced.** The absence holds in every reading at p < 0.05. Main reading: the |MI difference| of 12,271 pairs of 215 cells against their distance has a major-axis slope of -1.4e-05 per um, p = 0.818 (bootstrap) and 0.870 (shuffle); the difference in disparity preference of 7,952 pairs of 169 cells with DSI > 0.1 against distance gives circ_corrcl rho = 0.010, p = 0.690. Across the other readings p is 0.211 or more. The pairs are not independent, so these p-values, like the paper's, are optimistic; it does not change the absence.

### fig4d-pv-od-matches-local-50um, fig4f-pv-od-spatial-decay, fig4f-od-length-constant

> PV+ interneuron ocular dominance closely matches the average ocular dominance of local PV- cells
> within 50 um (mean slope 0.78 +/- 0.25 SE, n = 86, bootstrapped PCA).

> The match between PV+ ocular dominance and the local population average decreases with distance
> and is significant against shuffled populations only for the nearest populations (50 and 75 um,
> p < 0.01).

> An exponential fit to the PV+ ocular dominance slopes across radii gives a spatial length
> constant of 71.4 um (r2 = 0.80).

Each row: per radius, n, mean bootstrap slope +/- SE, the shuffled slopes' mean, and p one-sided
(p1) and two-sided (p2) against the shuffles. Axis (1a) individual on population, (1b) population
on individual.

In [ ]:
OD_PV = {}
for i, (name, cfg) in enumerate(OD_CONFIGS.items()):
    for axis in "ab":
        res = run_od(True, cfg, axis, 500 + 10 * i + (axis == "b"))
        OD_PV[(name, axis)] = res
        show(res, f"{name} ({'1' + axis})")
print()
r_ = np.array(pvc10.RADII, float)
print("exponential fits y = a exp(-r / lambda) to the five mean slopes (unweighted; weighted by SE):")
for (name, axis), res in OD_PV.items():
    if all("slope" in r for r in res):
        y = np.array([r["slope"] for r in res]); se = np.array([r["se"] for r in res])
        a, lam, r2 = pvc10.exp_fit(r_, y)
        aw, lamw, r2w = pvc10.exp_fit(r_, y, sigma=se)
        print(f"  {name:28s} 1{axis}: lambda {lam:8.1f} um, r2 {r2:6.2f}; weighted lambda {lamw:8.1f}, r2 {r2w:6.2f}")
    else:
        print(f"  {name:28s} 1{axis}: a radius has fewer than 3 cells, no fit")
print()
print("criteria per reading (p1 one-sided, p2 two-sided):")
for (name, axis), res in OD_PV.items():
    if not all("slope" in r for r in res):
        print(f"  {name:28s} 1{axis}: a radius has fewer than 3 cells")
        continue
    s50 = res[0]
    sig1 = [r["radius"] for r in res if r["p_one"] < 0.01]
    sig2 = [r["radius"] for r in res if r["p_two"] < 0.01]
    sig05 = [r["radius"] for r in res if r["p_one"] < 0.05]
    largest = s50["slope"] == max(r["slope"] for r in res)
    print(f"  {name:28s} 1{axis}: fig4d slope within {abs(s50['slope'] - 0.78) <= 0.25}, count within "
          f"{abs(s50['n'] - 86) <= 8.6}, p1 < 0.01 {s50['p_one'] < 0.01} | fig4f p1 < 0.01 at {sig1}, p2 < 0.01 at {sig2}, "
          f"p1 < 0.05 at {sig05}, 50 um largest {largest}; pattern {sig1 == [50, 75] and largest}, "
          f"only 50 um {sig1 == [50]}, pattern at p < 0.05 only {sig05 == [50, 75] and largest}")

**Result: not reproduced.** In the main reading with population on individual (reading 1b), the 50 um slope is 0.56 +/- 0.28, within the paper's 0.78 +/- 0.25, and it exceeds the shuffled slopes at p = 0.015 (one-sided) or 0.032 (two-sided), not at the p < 0.01 the criterion asks; the count is 18 PV+ cells against 86 +/- 8.6, from 3 of the 6 animals. Reading 1a (individual on population) gives 3.87 +/- 27.60, outside. Among the other readings N gives 0.40 +/- 0.27 (joint) and 0.52 +/- 0.39 (mono) from 5 cells each, and W gives 0.20 +/- 0.07 from 28 cells; none is significant at p < 0.01. The criterion has only a reproduced case, and the criteria check above shows the count failing under every reading, and the slope and p < 0.01 never met together (the one reading with p < 0.01 at 50 um, 7b with 1b, gives 1.42 +/- 1.94 from 17 cells), so by the criterion the claim is not reproduced. The outcome was partially reproduced until the third verification round, which agreed that the criterion gives not reproduced.

**Result: partially reproduced.** In the main reading (1b) the slopes at 50, 75, 100, 125 and 150 um are 0.56, 0.45, 0.13, 0.62 and -0.11, with one-sided shuffle p = 0.015, 0.091, 0.814, 0.009 and 0.989: neither 50 nor 75 um is significant at p < 0.01, 125 um is, and the 50 um slope is not the largest. At p < 0.05, 50 and 125 um are significant, which is not the pattern either. Reading 1a gives no slope significant at p < 0.01. No reading gives the full pattern (criteria check above). Under square pixels (7b) with population on individual (1b), only the 50 um slope is significant: 1.42 +/- 1.94 from 17 cells, p1 = 0.003 and p2 = 0.003, with p1 from 0.478 to 0.996 and p2 from 0.011 to 0.986 at 75 to 150 um. That is the criterion's partial case ("only the 50 um slope is significant"), under one reading as the criterion allows. The 50 um mean slope has an SE of 1.94, so the case rests on a fragile value, and p2 at 125 um (0.011) is just above 0.01, so it holds only narrowly under the two-sided reading. The shuffle test compares the mean of the bootstrap slopes with single shuffled slopes, as the Methods describe; with 17 cells that mean may sit above the slope of the data, which would make p1 too favourable. The slope of the data is not printed here. The outcome was not reproduced until the third verification round, which agreed with the change.

**Result: not reproduced.** Fitted to the main reading's five slopes (1b), the exponential gives lambda = 79.0 um with r2 = 0.31 (tolerance 64.3 to 78.5 um and 0.72 to 0.88); weighted by SE, 36.1 um and r2 0.11. Reading 1a gives no usable fit (lambda in the millions of um). Across all readings, r2 is at most 0.62 (L with square pixels, lambda 10.3 um).

### fig4ef-pvminus-od-unrelated

> Individual PV- ODI and local population averages are generally unrelated; there is a weak but
> significant relation within 50 um (mean slope 0.10 +/- 0.02, n = 717, p < 0.01), and none at other
> distances (p > 0.5), with no spatial dependence.

In [ ]:
OD_PVM = {}
for i, (name, cfg) in enumerate(OD_CONFIGS.items()):
    for axis in "ab":
        res = run_od(False, cfg, axis, 600 + 10 * i + (axis == "b"))
        OD_PVM[(name, axis)] = res
        show(res, f"{name} ({'1' + axis})")
print()
print("criteria per reading: 50 um slope within 0.10 +/- 0.02, count within 717 +/- 71.7, p1 < 0.01; "
      "radii 75 to 150 um with p1 or p2 < 0.01; largest bootstrap SE")
for (name, axis), res in OD_PVM.items():
    ok = [r for r in res if "slope" in r]
    s50 = res[0]
    far = [r["radius"] for r in ok[1:] if min(r["p_one"], r["p_two"]) < 0.01]
    print(f"  {name:28s} 1{axis}: 50 um slope within {abs(s50['slope'] - 0.10) <= 0.02}, count within "
          f"{abs(s50['n'] - 717) <= 71.7}, p1 < 0.01 {s50['p_one'] < 0.01}; significant at {far or 'no larger radius'}; "
          f"largest SE {max(r['se'] for r in ok):.2f}")

**Result: partially reproduced, resting on the assumption about the `binocmonocinds` lists and on reading 3b.** Under reading 1a (individual on population) the slopes are not usable (6.70 +/- 6.19 at 50 um, SEs up to 419.80), so the outcome is judged on 1b, which gives slopes of the size the paper reports. Under 1b with a cell dropped from all radii when one radius has fewer than 3 neighbours (3b), the 50 um slope is 0.09 +/- 0.04, within 0.10 +/- 0.02, but with one-sided p = 0.013, not the p < 0.01 asked, and from 127 PV- cells against 717; no radius from 75 to 150 um is significant at p < 0.01 (p1 0.165 to 0.953, p2 0.105 to 0.561). The absence holds and the 50 um values fall short: the criterion's partial case. W with 1b gives the same case (50 um 0.15 +/- 0.04, p1 = 0.233 with joint cells; no larger radius significant). In the main reading with 1b the 50 um slope is 0.17 +/- 0.06 (outside 0.08 to 0.12) and significant (p1 = 0.002), but 100 um is significant too (p1 = 0.002), so the absence fails there; that reading alone would give not reproduced.

### figs6ab-pvminus-pv-like-od

> PV- cells with ODI similar to PV+ neurons (-0.5 < ODI < 0) show no spatial dependence of ocular
> dominance, indistinguishable from shuffled populations (p > 0.10).

A claim of absence at p < 0.10 (two-sided, p2, as "indistinguishable" has no direction; p1 is
shown). Populations (a) unrestricted, (b) restricted to the same ODI range.

In [ ]:
like = lambda o: (o > -0.5) & (o < 0)
S6 = {}
for i, (name, cfg) in enumerate(OD_CONFIGS.items()):
    for pr in ("a", "b"):
        for axis in "ab":
            res = run_od(False, cfg, axis, 700 + 10 * i + 2 * (pr == "b") + (axis == "b"), ind_filter=like,
                         pop_filter=like if pr == "b" else None)
            S6[(name, pr, axis)] = res
            show(res, f"{name} pop {pr} ({'1' + axis})")
print()
print("per reading: radii with p2 < 0.10, and the largest bootstrap SE (far above 1 marks an unstable orientation)")
for (name, pr, axis), res in S6.items():
    ok = [r for r in res if "slope" in r]
    print(f"  {name:28s} pop {pr} 1{axis}: {[r['radius'] for r in ok if r['p_two'] < 0.10]}, largest SE {max(r['se'] for r in ok):.2f}")

**Result: reproduced, resting on the assumption about the `binocmonocinds` lists and on two further choices: unrestricted populations (reading a) and, for them, reading 1a.** The claims file leaves open whether the populations are restricted (b) as well as the individual cells; this attempt takes (a), as the Results restrict the individual cells ("Limiting our analysis to PV- cells") and say nothing of the populations. Judged at the paper's p < 0.10 on the two-sided shuffle p (p2), the main reading with (a) and 1a gives no radius below 0.10 (p2 0.236, 0.338, 0.105, 0.297, 0.541; 47 to 57 cells), and 1a is the stable orientation for (a) (largest SE 2.20, against 3117.32 under 1b). With restricted populations (b) the stable orientation is 1b (SEs 0.04 to 0.06), and there 75, 100 and 125 um give p2 = 0.051, 0.036 and 0.019, so under (b) the claim would not be reproduced. The 1a values for (b) (p2 0.288 to 0.974) come from slopes with SEs of 4.29 to 308.00 and do not support the outcome. With (a) and 1a, N with joint cells also gives no p2 below 0.10 (lowest 0.306), but N with monocular cells (100 um, p2 = 0.002) and W with joint cells (75 and 100 um, p2 = 0.036 and 0.020) do not, so the outcome depends on the cell set as well.

### figs4-same-plane-control

> PV+ interneurons still match population ocular dominance at small radii when neighbours in the
> same focal plane are excluded, and PV+ cells also show a distance-dependent relation with
> populations drawn from their own focal plane only.

In [ ]:
for i, (name, cfg) in enumerate(OD_CONFIGS.items()):
    for plane in ("other", "same"):
        for axis in "ab":
            res = run_od(True, cfg, axis, 800 + 10 * i + 2 * (plane == "same") + (axis == "b"), plane=plane)
            show(res, f"{name} {plane} plane ({'1' + axis})")

**Result: not reproduced.** (A) With neighbours in the individual's own plane excluded, the main reading gives no slope significant at p < 0.01 at 50 or 75 um (1b: 0.16, p = 0.483; 0.74, p = 0.020; 1a: p 0.306 and 0.820). (B) Within the individual's plane, the 1b slopes (0.57, 0.38, 1.07, 0.35, 0.74) do not decrease from 50 to 150 um and the 50 um slope is not significant (p = 0.513). No reading in the table gives a 50 or 75 um slope significant at p < 0.01 in (A).

### fig4i-pv-disparity-matches-local, fig4k-pv-disparity-length-constant

> Despite their weak selectivity, PV+ interneurons' disparity preferences are similar to those of
> nearby populations, significantly correlated with the 50 um population vector angle
> (shuffle-corrected r = 0.19 +/- 0.07 SE, n = 63).

> PV+ disparity preference shows a spatial dependence in its correlation with population vector
> angles; an exponential fit gives a length constant of 78.7 um (r2 = 0.63).

Individuals with DSI >= 0.1 and 50 um from the edges; populations of visually responsive PV- cells
(8a) or those with DSI >= 0.1 (8b); correlation (9a) as printed in the Methods or (9b) CircStat
circ_corrcc. Per radius: n, the shuffle-corrected value +/- bootstrap SE, the true value, and p of
the true value against the shuffles (one-sided).

In [ ]:
DISP_CONFIGS = {"main (L, 7a, 8a)": dict(resp="L", den="joint", sq=False, pop_dsi=False),
                "N (neuropil test), joint": dict(resp="N", den="joint", sq=False, pop_dsi=False),
                "N, bino": dict(resp="N", den="bino", sq=False, pop_dsi=False),
                "W (no neuropil test), joint": dict(resp="W", den="joint", sq=False, pop_dsi=False),
                "W, bino": dict(resp="W", den="bino", sq=False, pop_dsi=False),
                "L, square pixels (7b)": dict(resp="L", den="joint", sq=True, pop_dsi=False),
                "L, populations DSI >= 0.1 (8b)": dict(resp="L", den="joint", sq=False, pop_dsi=True)}


def run_disp(ind_pv, cfg, corr, seed, ind_filter=None, pop_filter=None):
    R = resp_sets(T, cfg["resp"])
    base = R[cfg["den"]] & np.isfinite(T["dsi"])
    ind = base & (T["pv"] == ind_pv) & edge[cfg["sq"]] & (T["dsi"] >= 0.1)
    pop = base & ~T["pv"]
    if cfg["pop_dsi"]:
        pop &= T["dsi"] >= 0.1
    if ind_filter is not None:
        ind &= ind_filter(T["dsi"])
    if pop_filter is not None:
        pop &= pop_filter(T["dsi"])
    return pvc10.disp_local(T, np.nonzero(ind)[0], np.nonzero(pop)[0], T["dsi"], T["pref"], pos[cfg["sq"]],
                            np.random.default_rng(seed), corr=corr)


def show_d(res, label):
    cells = []
    for r in res:
        if "corr" not in r:
            cells.append(f"{r['radius']}: n {r['n']}")
            continue
        sig = "*" if r["p_one"] < 0.05 else ""
        sig += "+" if r["corrected"] > 1.96 * r["se"] else ""
        cells.append(f"{r['radius']}: n {r['n']} {r['corrected']:.2f}+/-{r['se']:.2f} (r {r['corr']:.2f}) p {r['p_one']:.3f}{sig}")
    print(f"{label:44s} | " + " | ".join(cells))


print("* p < 0.05 against the shuffles (reading a); + corrected value above 1.96 SE (reading b)")
D_PV = {}
for i, (name, cfg) in enumerate(DISP_CONFIGS.items()):
    for corr in ("printed", "corrcc"):
        res = run_disp(True, cfg, corr, 900 + 10 * i + (corr == "corrcc"))
        D_PV[(name, corr)] = res
        show_d(res, f"{name} ({'9a' if corr == 'printed' else '9b'})")
print()
print("exponential fits to the five shuffle-corrected values (unweighted; weighted by SE):")
for (name, corr), res in D_PV.items():
    if all("corr" in r for r in res):
        y = np.array([r["corrected"] for r in res]); se = np.array([r["se"] for r in res])
        a, lam, r2 = pvc10.exp_fit(r_, y)
        aw, lamw, r2w = pvc10.exp_fit(r_, y, sigma=se)
        dec = "decreases 50 to 150 um" if y[0] > y[-1] else "does not decrease 50 to 150 um"
        print(f"  {name:36s} {corr:7s}: lambda {lam:8.1f} um, r2 {r2:6.2f}; weighted lambda {lamw:8.1f}, r2 {r2w:6.2f}; {dec}")
    else:
        print(f"  {name:36s} {corr:7s}: a radius has fewer than 3 cells, no fit")

**Result: not reproduced.** In the main reading 9 PV+ cells with DSI >= 0.1 have a 50 um population; the shuffle-corrected correlation is -0.04 +/- 0.20 (printed formula, 9a; p = 0.570) or -0.63 +/- 0.62 (CircStat, 9b; p = 0.954), against 0.19 +/- 0.07 with 63 cells. No reading gives a 50 um value that is significant under either test of significance; the one significant value in the main rows is at 100 um (9b, p = 0.039).

**Result: not reproduced.** The main reading's corrected correlations do not decrease from 50 to 150 um (9a: -0.04, -0.17, 0.22, 0.05, -0.01), and the fits give lambda = 20.7 um with r2 = 0.05 (9a) and 12.2 um with r2 = 0.49 (9b), against 78.7 um and 0.63. No reading has lambda and r2 within tolerance.

### fig4jk-pvminus-disparity-weak

> PV- neurons' disparity preferences show little relation to population biases; the
> shuffle-corrected circular correlation is significant only for the nearest neighbours (50 um,
> r = 0.05 +/- 0.02, n = 1,090).

In [ ]:
D_PVM = {}
for i, (name, cfg) in enumerate(DISP_CONFIGS.items()):
    for corr in ("printed", "corrcc"):
        res = run_disp(False, cfg, corr, 1000 + 10 * i + (corr == "corrcc"))
        D_PVM[(name, corr)] = res
        show_d(res, f"{name} ({'9a' if corr == 'printed' else '9b'})")

**Result: not reproduced.** In the main reading the 50 um corrected correlation for 130 PV- cells is -0.02 +/- 0.05 (9a, p = 0.658) or -0.04 +/- 0.09 (9b, p = 0.666), against 0.05 +/- 0.02 with 1,090 cells, and it is not significant, so neither the values nor the pattern of significance hold. No radius is significant in the main rows.

### figs6cd-pvminus-pv-like-disparity

> The lack of relation between PV- disparity preference and local populations is more evident for
> PV- cells with disparity tuning comparable to PV+ interneurons (0.1 < DSI < 0.2).

A claim of absence at p < 0.05: individuals PV- with 0.1 < DSI < 0.2; populations (a) unrestricted,
(b) restricted to the same range.

In [ ]:
dlike = lambda d: (d > 0.1) & (d < 0.2)
for i, (name, cfg) in enumerate(DISP_CONFIGS.items()):
    for pr in ("a", "b"):
        for corr in ("printed", "corrcc"):
            res = run_disp(False, cfg, corr, 1100 + 10 * i + 2 * (pr == "b") + (corr == "corrcc"), ind_filter=dlike,
                           pop_filter=dlike if pr == "b" else None)
            show_d(res, f"{name} pop {pr} ({'9a' if corr == 'printed' else '9b'})")
print(f"notebook time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

**Result: reproduced, resting on the assumption about the `binocmonocinds` lists, and on unrestricted populations (a) and the printed formula (9a) as the paper's test.** With individuals restricted to 0.1 < DSI < 0.2, populations unrestricted (a) and the printed formula (9a), the main reading gives no radius significant at p < 0.05 under either test of significance (p 0.066 to 0.777 against the shuffles; no corrected value above 1.96 SE, the largest 0.11 +/- 0.08 at 125 um), and the same holds for N and W with joint or binocular cells. The other readings do not all agree: with CircStat (9b) 100 um gives p = 0.047 in the main reading (and 75 or 150 um is significant in four of the six other configurations), and with populations restricted (b) 150 um gives p = 0.023 (9a) and 0.002 (9b). The outcome rests on (a) and (9a), the Methods' printed formula, as the paper's test.

## Summary

| Claim | Outcome |
|---|---|
| fig1e-pv-tdtomato-colocalization | blocked, data_missing (histology) |
| cell-counts-responsive | blocked, data_missing (identified cells of the six mice) |
| figs3-pv-distinct-from-neuropil, figs5-neuropil-no-relation | blocked, data_missing (neuropil patches; no non-responsive PV+ cells) |
| fig5-camkii-responsive, fig5e-camkii-od-unrelated, fig5f-camkii-disparity-unrelated | blocked, data_missing (CaMKIIa animals) |
| fig3abc-pv-more-binocular-fraction, mi-per-animal, figs6ab-pvminus-pv-like-od, figs6cd-pvminus-pv-like-disparity | reproduced (assumed) |
| fig3c-contralateral-bias, fig4ag-salt-and-pepper | reproduced |
| fig3-pv-lower-monocularity, fig3def-pv-weaker-disparity-selectivity, fig4ef-pvminus-od-unrelated | partially reproduced (assumed) |
| fig4f-pv-od-spatial-decay | partially reproduced |
| mi-dsi-relation, fig4d-pv-od-matches-local-50um | not reproduced (assumed) |
| pv-larger-dff-low-dsi, fig4f-od-length-constant, figs4-same-plane-control, fig4i-pv-disparity-matches-local, fig4k-pv-disparity-length-constant, fig4jk-pvminus-disparity-weak | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

Changed after the third verification round, which agreed with both: fig4d-pv-od-matches-local-50um
from partially reproduced to not reproduced (the criterion has no partial case), and
fig4f-pv-od-spatial-decay from not reproduced to partially reproduced (only the 50 um slope is
significant under readings 7b and 1b, 1.42 +/- 1.94, p = 0.003, a fragile value).

How the analysis differs from the paper's:

- The per-cell data come from structs outside the published folder, for mice 8 to 10 (and the
  PV+ cells of mouse 7), against the paper's 6 mice. Counts of cells in the analyses are therefore
  about half the paper's or less (for example 18 PV+ cells at 50 um against 86), which bears on
  every test of significance.
- The visually responsive cells are taken as the `binocmonocinds` lists (an assumption); the
  Methods' criterion is also applied with and without a neuropil test, and the outcomes that
  differ under those cell sets are marked as assumed. The lists decide every outcome; the other
  sets are shown as the sensitivity to this choice.
- ODI uses blank-subtracted responses with negatives set to zero, the form the stored DSI takes.
- The local analysis runs the claims file's open readings; reading 1a (individual on population)
  gives major-axis slopes with bootstrap SEs far above 1 in most cases, as population averages vary
  little, while 1b gives slopes of the size the paper reports.

What was found about the data set, from the outputs above:

- The published folder holds raw movies (green and red), stimulus tables and plane shifts for 58
  planes of 6 mice, and 50 example masks for one plane. It holds none of the per-cell data that its
  descriptor says it contains, and no masks or PV+/PV- labels for the paper's cells.
- The folder `original/to_remove/`, not in the file list or checksums, holds those per-cell data
  as MATLAB structs, one entry per cell: for mice 8, 9 and 10 (PV+ and PV-) and mouse 7 (PV+ only);
  the three mouse 7 PV- files are truncated, and mice 5 and 6 have index lists only. Their `Ftrace`
  equals the mask mean of the published movie exactly (9 cells of mouse 8, depth 205), and their
  trial counts per condition equal the stim files' repeats or one fewer. The stored DSI is the vector strength of blank-subtracted
  responses with negatives set to zero for all 578 cells; the stored responsiveness index matches
  its recomputation from `peaks` for 279 of 578 cells only. The structs hold only cells already
  selected (every PV+ cell passes the binocular index), so the paper's counts of identified cells
  cannot be checked. Nothing documents the structs, the `binocmonocinds` lists or their fields;
  the mouse 7 plane shift is stored as (-70, 157) where stats.mat gives (157, -70).
- The CaMKIIa-nls-tdTomato mice, the neuropil patches and the anti-PV histology are not in the data
  set; the structs hold one neuropil region per plane.
- The descriptor gives frames of 255 by 456 pixels; the files hold 256 by 455, as the paper says.
- No frame rate is given. At the paper's 30 Hz, the onset intervals give cycles of 3.93 s in
  mouse 5 and 4.37 to 4.43 s in mice 7 to 10, against the 4.5 s and 5 s of the stimulus plus a 3 s
  blank in the Methods; either the frame rate or the blank differs in five of six mice.
- In four planes the last stimulus cycle runs past the last frame. Eight planes have fewer than the
  6 to 7 repeats the paper states (all six planes of mouse 10 have 4 or 5), and mouse 10 has a
  plane at 125 um, above the Methods' 150 um.
- No recording was found deposited twice.
- On the example plane, stimulus trials differ from blank trials after onset, and the difference
  falls when onsets are moved, so the stimulus table is consistent with the movie's clock there.